In [ ]:
#imports
#python
import os, sys
import random
import itertools
from itertools import combinations
import scipy
from datetime import datetime
import re
from collections import defaultdict, OrderedDict

# image and data processing
import torch
from torchvision.models import alexnet
from torchvision import transforms as T
from PIL import Image , ImageOps
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.lines as mlines
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
import scipy.io
import scipy.optimize as optimize
from scipy.linalg import block_diag
from scipy.spatial.distance import cdist
from munkres import Munkres, print_matrix
#qiskit and d-wave
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo
#
from blueqat import Circuit
from blueqat.pauli import qubo_bit as q

from __future__ import annotations



In [ ]:
#search for and find the files
def path_joiner(image_name, root_dir = None): #recursive search for the image
    if root_dir is None:
        root_dir = os.getcwd()
    for dirpath , _, filenames in os.walk(root_dir):
        if image_name in filenames:
            return os.path.join(dirpath, image_name)

In [ ]:
#Keypoints extraction
def keypoint(image_path, max_points=None): # extract a desired number of keypoints from the images
    keypoints1 = scipy.io.loadmat(image_path)["pts_coord"]
    if max_points is not None:
        keypoints1 = keypoints1[:, :max_points]
    return keypoints1
def keypoints_dict(image_names: list, max_points: int):
    keypoints = {}
    for image_name in image_names:
        base_name = os.path.splitext(image_name)[0]
        full_path = path_joiner(image_name)
        if full_path:
            kps = keypoint(full_path, max_points)
            keypoints[base_name] = kps
        else:
            print(f"[Warning] image not found: {image_name}")
    return keypoints
def keypoints_list(image_paths: list, max_points: int):
    keypoints = []
    for image_path in image_paths:
        keypoints1 = keypoint(image_path, max_points)
        keypoints.append(keypoints1)
    return keypoints

In [ ]:
#Feature extraction using AlexNet

def alexnet_feature_extractor(layer= 'conv4'):
    model = alexnet(pretrained=True)
    model.eval()
    if layer == 'conv4':
        return torch.nn.Sequential(*list(model.features)[:10])
    elif layer == 'conv5':
        return torch.nn.Sequential(*list(model.features)[:12])
    else:
        raise ValueError("Invalid layer. Choose 'conv4' or 'conv5'.")
def extract_features(keypoints_dict, patch_size=64, layer='conv4'):
    feature_extractor = alexnet_feature_extractor(layer)
    transform = T.Compose([
        T.Resize((227, 227)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_extractor.to(device)
    features = {}

    for image_name, keypoints in keypoints_dict.items():
        img_path = path_joiner(image_name + '.png')
        img = Image.open(img_path).convert('RGB')
        # img = Image.open(img_path).convert('L')
        img_np = np.array(img)
        feature_list = []

        for (x, y) in keypoints.T:
            half = patch_size // 2

            # Pad the image if keypoint is near the edge
            padding = (half, half, half, half)  # left, top, right, bottom
            padded_img = ImageOps.expand(img, border=padding, fill=0)

            # shift keypoint for padding the edges
            x_padded = x + half
            y_padded = y + half

            # safely crop the full patch_size
            patch = padded_img.crop((x_padded - half, y_padded - half,
                                    x_padded + half, y_padded + half))
            # patch = patch.convert('RGB')  # duplicate L channel into R, G, B for alexnet
            patch_tensor = transform(patch).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = feature_extractor(patch_tensor)
                feat = feat.mean(dim=[2, 3]) # to flatten the output dimensions
            feature_list.append(feat.squeeze().cpu().numpy())

        features[image_name] = np.stack(feature_list)
    return features

In [ ]:
#features have been extracted, now we need to calculate the distance matrix
def pairwise_permutations(features_dict, pm_size: int) -> dict: # compute the P_ij
    P = {}
    image_list = list(features_dict.keys())

    for i in range(len(image_list)):
        key0= f"P{i+1}{i+1}"
        P[key0] = np.eye(pm_size)
        for j in range(i + 1, len(image_list)):
            img1 = image_list[i]
            img2 = image_list[j]

            feats1 = features_dict[img1]
            feats2 = features_dict[img2]
            similarity = cosine_similarity(feats1, feats2) # equivalent to cosine similarity, since the features are normalized
            #print("this is the similarity matrix")
            print(similarity)
            cost_mat = (1 - similarity)
            row_ind, col_ind = linear_sum_assignment(cost_mat)
            n = len(row_ind)
            perm_matrix = np.zeros((n, n), dtype=int)
            perm_matrix[row_ind, col_ind] = 1
            key1 = f"P{i+1}{j+1}"
            P[key1] = perm_matrix
            key2 = f"P{j+1}{i+1}"
            P[key2] = perm_matrix.T
    return P

In [ ]:


#QUBO form maker with X1 fixed to identity
def qubo_form_maker_fixed_gauge(P: dict,
                                num_views: int,
                                num_keypoints: int,
                                penalty: float = 2.5) -> tuple[np.ndarray, np.ndarray]:
    """
    Construct the QUBO matrix Q and linear vector s for permutation synchronization,
    fixing the gauge by setting the first absolute permutation matrix X1 = I.

    Args:
        P: Dictionary of pairwise permutations.  Keys are strings like "P12", "P21", "P34", "P43", etc.
           Each value is an (n x n) permutation matrix.  It must include both P_{ij} and P_{ji} = (P_{ij})^T
           for every i != j, and P_{ii} = I_n for each i.
        num_views: Total number of views (m).
        num_keypoints: Number of keypoints per view (n).
        penalty:   λ value for the row/column‐sum penalty term (default 2.5).

    Returns:
        (Q, s) where Q is an ((m-1)*n^2 x (m-1)*n^2) numpy array, and s is a vector of length (m-1)*n^2.
        Those define the QUBO:
            minimize  x^T Q x  +  s^T x
        over x ∈ {0,1}^{(m-1)*n^2}, representing vec(X2), …, vec(Xm),
        subject to each X_i being a permutation (enforced via penalty).
    """
    m = num_views
    n = num_keypoints
    N = n * n
    # If there's only one view, there's nothing to optimize.
    if m <= 1:
        return np.zeros((0, 0)), np.zeros((0,))

    # We fix X1 = I_n.  The unknowns are X2,...,Xm → total of (m-1) blocks of size n^2.
    m_opt = m - 1
    num_vars = m_opt * N

    # Initialize Q' (the raw bilinear terms among X2..Xm) and the linear vector s from interactions with X1.
    Q_prime = np.zeros((num_vars, num_vars))
    s_linear = np.zeros(num_vars)

    # Precompute vec(I_n) as x1:
    x1 = np.zeros((N,))
    for k in range(n):
        x1[k * n + k] = 1  # places 1 at indices [0, n+1, 2n+2, ..., (n-1)*n+(n-1)]

    # Helper to compute the block index range for view i (1-based indexing):
    #   If i > 1, then opt_index = (i-2), and that block covers [opt_index*N : (opt_index+1)*N).
    def var_slice(view_index_1_based: int):
        # Only valid for view_index ≥ 2.  Returns (start, end) in [0..num_vars)
        opt_idx = view_index_1_based - 2
        start = opt_idx * N
        end = start + N
        return start, end

    # 1) Build Q' among unknowns X2..Xm.
    #    For every unordered pair i<j, both > 1:
    #      - read Pij = P[f"P{i}{j}"] and Pji = P[f"P{j}{i}"] = (Pij)^T
    #      - form block_{ij} = kron(I_n, Pij), block_{ji} = kron(I_n, Pji)
    #      - subtract block_{ij} from Q'[i_block, j_block], subtract block_{ji} from Q'[j_block, i_block].
    for i in range(2, m + 1):          # i = 2..m
        for j in range(i + 1, m + 1):  # j = i+1..m
            key_ij = f"P{i}{j}"
            key_ji = f"P{j}{i}"
            if key_ij not in P or key_ji not in P:
                raise KeyError(f"Both '{key_ij}' and '{key_ji}' must be present in P.")
            Pij = P[key_ij]
            Pji = P[key_ji]
            if Pji.shape != (n, n) or Pij.shape != (n, n):
                raise ValueError(f"P[{key_ij}] and P[{key_ji}] must be {n}×{n} matrices.")
            # Build Kronecker blocks:
            block_ij = np.kron(np.eye(n), Pij)   # shape (n^2, n^2)
            block_ji = np.kron(np.eye(n), Pji)   # = (block_ij)^T if Pji = Pij^T

            # Place them into Q':
            start_i, end_i = var_slice(i)
            start_j, end_j = var_slice(j)
            # Subtract block_ij from Q'[i_block, j_block]
            Q_prime[start_i:end_i, start_j:end_j] -= block_ij
            # Subtract block_ji from Q'[j_block, i_block]
            Q_prime[start_j:end_j, start_i:end_i] -= block_ji

    # 2) Build the linear terms from interactions between X1=I_n and each X_j for j=2..m.
    #    In the paper: each pairwise term is -2 * Tr(P_{1j}^T * I_n * X_j^T) = -2 * vec(X_j)^T vec(P_{1j}^T).
    #    Hence the linear coefficient (a length-n^2 vector) is -2 * vec(P_{1j}^T) = -2 * (x1^T @ (I tensor P_{1j})).
    for j in range(2, m + 1):
        key_1j = f"P1{j}"
        if key_1j not in P:
            raise KeyError(f"'{key_1j}' must be present in P (pairwise with view 1).")
        P1j = P[key_1j]
        if P1j.shape != (n, n):
            raise ValueError(f"P['{key_1j}'] must be an {n}×{n} matrix.")
        # Build the Kronecker block:
        block_1j = np.kron(np.eye(n), P1j)  # shape (n^2, n^2)

        # Compute vec(P1j^T)^T = vec(I * P1j)^T = x1^T @ (I tensor P1j).
        # Then multiply by -2 for the objective.
        linear_contrib = -2 * (x1 @ block_1j)  # shape (n^2,)

        # Add into the appropriate slice of s_linear (the block for view j).
        start_j, end_j = var_slice(j)
        s_linear[start_j:end_j] += linear_contrib

    # Note: We do not separately process (i,1) because we already accounted for all (1,j) with the -2 factor.
    #       We also ignore diagonal P_{ii} entirely since they add only constants.

    # 3) Build the A matrix and b vector for the row‐sum & col‐sum constraints on X2..Xm.
    num_constraints = m_opt * 2 * n
    A = np.zeros((num_constraints, num_vars), dtype=float)
    b = np.ones((num_constraints,), dtype=float)

    row_ctr = 0
    for view_idx_opt in range(m_opt):  # view_idx_opt = 0..(m-2) corresponds to original view = view_idx_opt+2
        offset = view_idx_opt * N
        # Row‐sum constraints: for each row r in 0..n-1, sum_{c=0..n-1} x[row*r + c] = 1
        for r in range(n):
            for c in range(n):
                var_idx = offset + r * n + c
                A[row_ctr, var_idx] = 1
            row_ctr += 1

        # Column‐sum constraints: for each column c in 0..n-1, sum_{r=0..n-1} x[r*n + c] = 1
        for c in range(n):
            for r in range(n):
                var_idx = offset + r * n + c
                A[row_ctr, var_idx] = 1
            row_ctr += 1

    # 4) Form the final QUBO:  Q_total = Q' + penalty (A^T A),   s_total = s_linear - 2penalty A^T b.
    Q = Q_prime + penalty * (A.T @ A)
    s = s_linear - 2 * penalty * (A.T @ b)

    return Q, s




In [ ]:
## needed one helper function

def to_float(x):
    """Return numbers even if x is a 0‑D/1‑D NumPy number. to be safer"""
    return float(np.asarray(x))

# QAOA and Hamiltonian
def build_blueqat_hamiltonian(Q: np.ndarray, s: np.ndarray):
    """Convert ``cost = xT Q x + sT x`` to a Blueqat PauliSum.

    * Diagonal of *Q* is absorbed into the linear part because *x_i² = x_i*.
    * Off‑diagonal is taken from upper‑triangle only.
    * All coefficients are cast to plain Python ``float`` so that Blueqat’s
      PauliTerm constructor never receives a NumPy scalar (this was the source
      of the previous ValueError).
    """
    Q = np.asarray(Q, dtype=float)
    s = np.asarray(s, dtype=float).ravel()
    n = Q.shape[0]

    H = 0  # PauliSum accumulator ------------------------------------------------

    # linear (includes the diagonal of Q)
    for i in range(n):
        coeff = to_float(s[i] + Q[i, i])
        if coeff:
            H += coeff * q(i)

    # quadratic (strictly upper‑triangle)
    for i in range(n):
        for j in range(i + 1, n):
            coeff = to_float(Q[i, j])
            if coeff:
                H += coeff * q(i) * q(j)

    return H.simplify()

In [ ]:
def make_expectation(H, time_evolutions, p: int, backend="quimb"):
    """Return ``f(params)`` that computes ⟨psi(beta,gamma)|H|psi(beta,gamma)⟩."""
    n_qubits = H.n_qubits

    def f(params):
        beta = params[:p]
        gamma = params[p:]
        print(f"Evaluating with beta={beta}, gamma={gamma}")

        c = Circuit(n_qubits).h[:]
        for layer in range(p):
            for evo in time_evolutions:
                evo(c, gamma[layer])
            c.rx(beta[layer])[:]
        
        result = c.run(backend=backend, hamiltonian=H)
        print(f"Expectation value: {result}")
        return result

    return f

In [ ]:
def run_qaoa(
    Q: np.ndarray,
    s: np.ndarray,
    p: int = 1,
    method: str = "Nelder-Mead",
    tol: float | None = 1e-3,
    random_seed: int | None = None,
    backend: str = "quimb",
):
    """Optimise p‑layer QAOA.

    Returns ``(best_expectation, scipy_result)``.
    """
    print("Starting QAOA optimization...")
    
    if random_seed is not None:
        print("Setting random seed...")
        np.random.seed(random_seed)

    print("Building Hamiltonian...")
    H = build_blueqat_hamiltonian(Q, s)

    print("Calculating time evolutions...")
    time_evos = [t.get_time_evolution() for t in H]

    print("Preparing expectation function...")
    f = make_expectation(H, time_evos, p, backend=backend)
    # Randomly initialize 2*p angles in [0, 2pi)
    init_angles = 2 * np.pi * np.random.rand(2 * p)

    print(f"Initializing angles {init_angles} and starting optimization...")
    init = 2 * np.pi * np.random.rand(2 * p)
    #applying optional bounds in case the method is a bounded one
    bounds = None
    if method in ("L-BFGS-B", "TNC", "SLSQP"):
        # Constrain beta ,gamma   [0, 2π]
        bounds = [(0, 2 * np.pi)] * (2 * p)
    res = optimize.minimize(f, init, method=method, tol=tol,bounds=bounds)

    if not res.success:
        raise RuntimeError(res.message)

    print("Optimization complete.")
    return res.fun, res


In [ ]:
#New permutation extraction method

def is_valid_permutation_vector(bit_arr: np.ndarray, num_matrices: int, n: int) -> bool:
    """
    Checks if a flat bit array represents a list of valid permutation matrices.

    Args:
        bit_arr: The flat array of 0s and 1s from the QAOA sample.
        num_matrices: The number of matrices encoded in the vector (m-1).
        n: The dimension of each matrix (n x n).

    Returns:
        True if all matrices are valid permutations, False otherwise.
    """
    N = n * n
    if len(bit_arr) != num_matrices * N:
        return False

    for i in range(num_matrices):
        start = i * N
        matrix = bit_arr[start : start + N].reshape((n, n))
        # Check if each row and each column sums to 1
        if not (np.all(matrix.sum(axis=1) == 1) and np.all(matrix.sum(axis=0) == 1)):
            return False
    return True






## Permutation extraction and accuracy calculator
def extract_permutation_matrices_fixed_gauge(
    sample: dict[int, int],
    num_views: int,
    num_keypoints: int
) -> dict[str, np.ndarray]:
    """
    Given a QAOA‐sampled bit‐assignment `sample` (mapping qubit_index → 0/1)
    for a QUBO built with X1 fixed to identity, reconstruct the absolute
    permutation matrices X1, X2, …, X_m.

    Args:
      sample: dict where keys are integers 0..((m-1)*n^2 - 1), values are 0 or 1.
              These correspond only to the flattened entries of X2..Xm, in order.
      num_views:  m, total number of views (including X1).
      num_keypoints:  n, the size of each permutation matrix (n×n).

    Returns:
      A dict mapping 'X1', 'X2', …, 'X_m' to their reconstructed n×n permutation matrices.
      X1 is always set to identity (np.eye(n)).  X2..Xm come from `sample`.
    """
    m = num_views
    n = num_keypoints
    N = n * n

    # First, verify that `sample` has exactly (m-1)*N keys 0..((m-1)*N - 1)
    num_vars_expected = (m - 1) * N
    if set(sample.keys()) != set(range(num_vars_expected)):
        raise ValueError(
            f"Expected sample keys 0..{num_vars_expected-1}, "
            f"but got keys {sorted(sample.keys())[:5]}..."
        )

    # Build a flat vector x_opt of length (m-1)*N
    # in ascending qubit order:
    x_opt = np.array([sample[i] for i in range(num_vars_expected)], dtype=int)

    # Initialize output dict
    matrices = {}

    # X1 = identity
    matrices["X1"] = np.eye(n, dtype=int)

    # For each view j = 2..m, extract the next N bits and reshape
    for j in range(2, m + 1):
        start = (j - 2) * N
        block = x_opt[start : start + N]
        matrices[f"X{j}"] = block.reshape((n, n))

    return matrices


def permutation_accuracy(
    pred: dict[str, np.ndarray] | list[np.ndarray],
    true: dict[str, np.ndarray] | list[np.ndarray]
) -> float:
    """
    Compute bitwise accuracy between predicted and true permutation matrices.
    Accuracy = 1 − (number of mismatched bits) / (total bits).

    Accepts either:
      - Two dicts with identical keys 'X1','X2',…,'Xm', each mapping to an n×n array.
      - Two lists of length m, each entry an n×n array, implicitly X1..Xm.

    Returns:
      A float in [0,1] giving fraction of matching entries across all matrices.
    """
    # Convert lists to dicts 'X1','X2',...
    if isinstance(pred, list):
        pred = {f"X{i+1}": np.array(pred[i], dtype=int) for i in range(len(pred))}
    if isinstance(true, list):
        true = {f"X{i+1}": np.array(true[i], dtype=int) for i in range(len(true))}

    # Ensure both have the same keys
    pred_keys = set(pred.keys())
    true_keys = set(true.keys())
    if pred_keys != true_keys:
        raise ValueError(f"Prediction keys {pred_keys} do not match ground‐truth keys {true_keys}")

    total_bits = 0
    mismatches = 0
    # Iterate over sorted keys 'X1','X2',...
    for key in sorted(pred.keys(), key=lambda k: int(k[1:])):
        Xp = np.array(pred[key], dtype=int)
        Xt = np.array(true[key], dtype=int)
        if Xp.shape != Xt.shape:
            raise ValueError(f"Shape mismatch for {key}: pred={Xp.shape}, true={Xt.shape}")
        total_bits += Xp.size
        mismatches += np.sum(Xp != Xt)

    if total_bits == 0:
        return 0.0
    return 1.0 - (mismatches / total_bits)

In [ ]:
##Functions for visualization(Just like the one in annealing)
def generate_n_colors(n):
    """Return a list of n visually distinct colors."""
    cmap = plt.cm.get_cmap("tab10", n)
    return [cmap(i) for i in range(n)]

def visualize_and_save_result(
    points_list, 
    permutation_matrices, 
    mat_files, 
    best_energy=None, 
    penalty_value=None, 
    output_dir="./results", 
    save_fig=True, 
    show_fig=True, 
    radius=5,
    dpi=120
):


    os.makedirs(output_dir, exist_ok=True)
    num_views = len(points_list)
    # --- Ensure keypoints are 2xN ---
    for i in range(len(points_list)):
        if points_list[i].shape[0] != 2:
            print(f"[Debug] Transposing keypoints for view {i} from shape {points_list[i].shape}")
            points_list[i] = points_list[i].T
    num_keypoints = points_list[0].shape[1]
    colors = generate_n_colors(num_keypoints)

    # --- Validate permutation matrices ---
    for k, X in permutation_matrices.items():
        if not (np.all(X.sum(axis=0) == 1) and np.all(X.sum(axis=1) == 1)):
            print(f"[Warning] {k} is not a valid permutation matrix! Row sums: {X.sum(axis=1)}, Col sums: {X.sum(axis=0)}")

    # --- Build matches (traces for each keypoint across all views) ---
    matches = [[] for _ in range(num_keypoints)]
    for i in range(num_keypoints):
        matches[i].append(i)  # X1: identity
    for view_idx in range(1, num_views):
        X = permutation_matrices[f'X{view_idx+1}']
        for i in range(num_keypoints):
            target = np.argmax(X[i])
            matches[i].append(target)

    # Loading the images
    images = []
    x_offsets = []
    x_offset = 0
    for mat_file in mat_files:
        img_path = path_joiner(mat_file.replace('.mat', '.png')) if not os.path.exists(mat_file.replace('.mat', '.png')) else mat_file.replace('.mat', '.png')
        img = Image.open(img_path).convert("RGB")
        images.append(img)
        x_offsets.append(x_offset)
        x_offset += img.size[0]

    total_width = sum(img.size[0] for img in images)
    max_height = max(img.size[1] for img in images)
    canvas = Image.new('RGB', (total_width, max_height), color=(255, 255, 255))
    for idx, img in enumerate(images):
        canvas.paste(img, (x_offsets[idx], 0))

    fig, ax = plt.subplots(figsize=(min(6*num_views, 20), 5))
    ax.imshow(canvas)
    ax.axis('off')

    for keypoint_idx, match in enumerate(matches):
        color = colors[keypoint_idx]
        prev_x, prev_y = None, None
        for view_idx, kp_idx in enumerate(match):
            pt = points_list[view_idx][:, kp_idx]
            x = x_offsets[view_idx] + pt[0]
            y = pt[1]
            circle = plt.Circle((x, y), radius, color=color, fill=True, zorder=3)
            ax.add_patch(circle)
            if prev_x is not None:
                line = mlines.Line2D([prev_x, x], [prev_y, y], color=color, linewidth=2, zorder=2)
                ax.add_line(line)
            prev_x, prev_y = x, y

    # Save figure and text file
    image_save_path, txt_save_path = None, None
    if save_fig:
        short_names = "_".join([os.path.splitext(os.path.basename(f))[0].lower() for f in mat_files])
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        base_filename = f"{short_names}_{timestamp}"
        image_save_path = os.path.join(output_dir, base_filename + ".png")
        txt_save_path = os.path.join(output_dir, base_filename + ".txt")
        fig.savefig(image_save_path, bbox_inches='tight', pad_inches=0.1, dpi=dpi)
        with open(txt_save_path, "w") as f:
            f.write(f"Images: {mat_files}\n")
            if penalty_value is not None: f.write(f"Penalty: {penalty_value}\n")
            if best_energy is not None: f.write(f"Energy: {best_energy}\n")
            f.write(f"Permutation Matrices:\n")
            for k, mat in permutation_matrices.items():
                f.write(f"{k}:\n{mat}\n\n")
        print(f"Saved to: {image_save_path}, {txt_save_path}")
    if show_fig:
        plt.show()
    else:
        plt.close(fig)
    return image_save_path, txt_save_path

In [ ]:
"""
    Run the full pipeline and creation of the problems from the categories just like the annealing
    Then do sampling and circuit building
"""


def compute_qubo_energy(bit_arr: np.ndarray, Q: np.ndarray, s: np.ndarray) -> float:
    """
    Compute E = x^T Q x + s^T x for a binary vector x.
    """
    x = bit_arr
    return float(x @ (Q @ x) + s @ x)


def sample_qaoa_solution(H, time_evos, Q: np.ndarray, s: np.ndarray,
                         p: int, beta: np.ndarray, gamma: np.ndarray,
                         shots: int = 500, backend: str = "quimb"):
    """
    Given optimal angles (beta, gamma), sample the QAOA state `shots` times,
    evaluate QUBO energy for each bitstring, and return the best sample (dict)
    plus its energy.

    Returns:
      best_sample_dict:  { qubit_index: 0 or 1 }  for the lowest-energy bitstring
      best_energy:      float, that bitstring QUBO energy
    """
    n_qubits = H.n_qubits

    # Build the final QAOA circuit
    c = Circuit(n_qubits).h[:]  # |+>^n
    for layer in range(p):
        gamma_parameter = float(gamma[layer])
        for evo in time_evos:
            evo(c, gamma_parameter)
        beta_parameter = float(beta[layer])
        c.rx(beta_parameter)[:]
    # Sample `shots` times
    # This returns a dict of {bitstring: count}
    samples = c.run(backend=backend, shots=shots)

    best_energy = float("inf")
    best_sample_dict = None

    # Iterate over each unique bitstring returned
    for bitstr, count in samples.items():
        # Convert bitstring (e.g. "010101") to array of ints [0,1,0,1,0,1]
        # Assume bitstr[0] corresponds to qubit 0, etc.
        bit_arr = np.array([int(ch) for ch in bitstr], dtype=int)
        energy = compute_qubo_energy(bit_arr, Q, s)
        if energy < best_energy:
            best_energy = energy
            # Build dict {qubit_index: bit} from bit_arr
            best_sample_dict = {i: int(bit_arr[i]) for i in range(n_qubits)}

    return best_sample_dict, best_energy

###########################################NEW SAMPLE FUNCTION

def find_best_valid_solution_from_samples(
    H_penalized, time_evos, Q_original, s_original,
    p: int, beta: np.ndarray, gamma: np.ndarray,
    m_opt: int, n: int, shots: int = 1000, backend: str = "quimb"
) -> tuple[dict | None, float | None]:
    """
    Samples the QAOA state and finds the lowest-energy *valid* permutation.

    Args:
        H_penalized: The full Hamiltonian including the penalty term.
        time_evos: Pre-calculated time evolutions for H_penalized.
        Q_original: The QUBO Q-matrix *without* penalties.
        s_original: The QUBO s-vector *without* penalties.
        p: QAOA depth.
        beta, gamma: The optimized angles.
        m_opt: The number of matrices being optimized (m-1).
        n: The dimension of the matrices (n x n).
        shots: Number of times to sample the circuit.

    Returns:
      A tuple (best_valid_sample, best_valid_energy), or (None, None) if no
      valid samples were found.
    """
    n_qubits = H_penalized.n_qubits
    c = Circuit(n_qubits).h[:]
    for layer in range(p):
        for evo in time_evos:
            evo(c, float(gamma[layer]))
        c.rx(float(beta[layer]))[:]
    
    samples = c.run(backend=backend, shots=shots)

    best_valid_energy = float("inf")
    best_valid_sample = None
    
    for bitstr, count in samples.items():
        bit_arr = np.array([int(ch) for ch in bitstr], dtype=int)
        
        if is_valid_permutation_vector(bit_arr, m_opt, n):
            # If valid, calculate its *original* energy using the classical QUBO formula
            energy = compute_qubo_energy(bit_arr, Q_original, s_original)
            if energy < best_valid_energy:
                best_valid_energy = energy
                best_valid_sample = {i: bit_arr[i] for i in range(len(bit_arr))}

    return best_valid_sample, best_valid_energy


##############################################################

# Problem creation and and solving through QAOA for the Willow dataset


categories = ["car", "duck", "winebottle", "motorbike"]
window_size = 3
num_keypoints = 3
penalty_value = 100.0

category_accuracies = {cat: [] for cat in categories}

def get_gt_absolute_permutations(points_list):
    """
    ground‐truth keypoints are manually annotated across views(so all the points permutation must correspond to identities)
    So each X_i = I_n.  Returns [I_n, I_n, …] of length m.
    """
    m = len(points_list)
    n = points_list[0].shape[1]
    return [np.eye(n, dtype=int) for _ in range(m)]


for category in categories:
    folder = f"./PF-dataset/{category}"
    all_mats = sorted([f for f in os.listdir(folder) if f.endswith(".mat")])
    num_images = len(all_mats)
    if num_images < window_size:
        print(f"Not enough .mat files in {folder} for window size {window_size}")
        continue

    num_problems = 3
    print(f"\nProcessing category '{category}' with {num_problems} problems...")

    for i in range(num_problems):
        group = all_mats[i : i + window_size]
        mat_paths = [os.path.join(folder, f) for f in group]

        # 1) Extract some keypoints per image, compute deep features
        points_dic = keypoints_dict(group, max_points=num_keypoints)
        features = extract_features(points_dic, patch_size=64, layer="conv4")
        P = pairwise_permutations(features, num_keypoints)

        # 2) Build QUBOs
        Q, s = qubo_form_maker_fixed_gauge(P, window_size, num_keypoints, penalty=penalty_value)
        Q_orig, s_orig = qubo_form_maker_fixed_gauge(P, window_size, num_keypoints, penalty=0.0)

        # 3) Run QAOA to optimize angles
        best_exp, opt_result = run_qaoa(
            Q,
            s,
            p=1,
            method="Nelder-Mead",
            tol=1e-3,
            random_seed=42,
            backend="quimb",
        )
        beta_opt = opt_result.x[:1]
        gamma_opt = opt_result.x[1:]

        # 4) Build Hamiltonian and time‐evolutions again for sampling
        H = build_blueqat_hamiltonian(Q, s)
        time_evos = [term.get_time_evolution() for term in H]

        # 5) Sample QAOA and find the best valid solution
        best_sample, best_energy = find_best_valid_solution_from_samples(
            H,
            time_evos,
            Q_orig,
            s_orig,
            p=1,
            beta=beta_opt,
            gamma=gamma_opt,
            m_opt=(window_size - 1),
            n=num_keypoints,
            shots=250,
            backend="quimb"
        )
        
        if best_sample is None:
            print(f"ATTENTION!!: No valid permutation matrices were found for {category}_{i}.")
            category_accuracies[category].append(0.0)
            continue

        # 6) Decode into absolute permutations X1..Xm
        matrices = extract_permutation_matrices_fixed_gauge(
            best_sample,
            window_size,
            num_keypoints
        )
        print(f"{category}_{i}_absolute matrices (via QAOA):")
        print(matrices)

        # 7) Visualize & compare to ground truth
        points_list = keypoints_list(mat_paths, num_keypoints)
        gt_matrices = get_gt_absolute_permutations(points_list)
        visualize_and_save_result(
            points_list=points_list,
            permutation_matrices=matrices,
            mat_files=mat_paths,
            best_energy=best_energy,
            penalty_value=penalty_value,
            output_dir="./results_qaoa",
            save_fig=True,
            show_fig=True,
            radius=6
        )

        # 8) Compute accuracy
        pred_list = [matrices[f"X{j+1}"] for j in range(window_size)]
        acc = permutation_accuracy(pred_list, gt_matrices)
        category_accuracies[category].append(acc)

        print(f"Problem {i+1}/{num_problems} | Files: {group} | Accuracy: {acc:.3f}")

print("\nDone! Mean accuracies per category:")
for cat in categories:
    if category_accuracies[cat]:
        print(f"{cat:10s}: {np.mean(category_accuracies[cat]):.3f}")
    else:
        print(f"{cat:10s}: No data")
